# 15 — Unit-Level Spatial Features
## RentSignal — Data Architecture Phase 3

**Goal:** Replace PLZ-level spatial features with point-level features computed from each unit's actual coordinates.

**Tables produced:**
| Table | File | Description |
|-------|------|-------------|
| `spatial_unit` | `data/processed/spatial_unit.parquet` | 8,250 rows × ~20 spatial features per unit |

**Two feature groups:**

### A. OSM Point Features (9 features, replacing PLZ centroid)
Computed from actual unit lat/lon → real distances/counts, not PLZ centroid approximations.
- `dist_transit_m`, `dist_park_m`, `dist_water_m`, `dist_school_m`
- `count_food_500m`, `count_food_1000m`, `count_shop_500m`, `count_shop_1000m`, `count_transit_1000m`

### B. Satellite Multi-Scale Features (9 features, replacing PLZ zonal stats)
Extracted from local Sentinel-2 rasters at 3 buffer scales per unit.
- `ndvi_100m`, `ndvi_250m`, `ndvi_500m`
- `ndwi_100m`, `ndwi_250m`, `ndwi_500m`
- `ndbi_100m`, `ndbi_250m`, `ndbi_500m`

**Data sources:**
- OSM POIs: Overpass API (re-queried, saved as GeoJSON for reuse)
- Satellite: Local TIF files (`data/processed/ndvi_berlin.tif`, `ndwi_berlin.tif`, `ndbi_berlin.tif`)

In [2]:
import sys, json, requests, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import rasterio
from rasterio.transform import rowcol
warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('..').resolve()
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'

# Load units with coordinates from Phase 2
units = pd.read_parquet(PROC_DIR / 'units.parquet')
has_coords = units['lat'].notna() & units['lon'].notna()
print(f'Units loaded: {len(units):,}')
print(f'With coordinates: {has_coords.sum():,} ({100*has_coords.mean():.1f}%)')
print(f'Coordinate sources: {units["coord_source"].value_counts().to_dict()}')

Units loaded: 8,256
With coordinates: 8,250 (99.9%)
Coordinate sources: {'listing': 5081, 'plz_ortsteil_centroid': 2283, 'kaggle_match': 745, 'plz_centroid': 116, 'title_mining': 25}


## Part A: OSM Point-Level Features

### A.1 — Download Berlin POIs via Overpass API

Query 6 POI categories from OpenStreetMap. Results saved as GeoJSON for reuse.

In [3]:
import time as _time

OVERPASS_URL = 'https://overpass-api.de/api/interpreter'
BERLIN_BBOX = '52.33,13.08,52.68,13.76'
POI_CACHE_DIR = PROC_DIR / 'osm_pois'
POI_CACHE_DIR.mkdir(exist_ok=True)

# POI queries — dropped sbahn (OSM tagging inconsistent, transit query covers it)
POI_QUERIES = {
    'ubahn': f'''[out:json][timeout:120];
(
  node["railway"="station"]["station"~"subway|light_rail"]({BERLIN_BBOX});
  node["railway"="station"]["network"~"U-Bahn"]({BERLIN_BBOX});
);
out center;''',

    'transit': f'''[out:json][timeout:120];
(
  node["railway"="station"]["station"~"subway|light_rail"]({BERLIN_BBOX});
  node["railway"="station"]["network"~"S-Bahn|U-Bahn|VBB"]({BERLIN_BBOX});
  node["railway"="halt"]({BERLIN_BBOX});
  node["public_transport"="station"]({BERLIN_BBOX});
  node["amenity"="bus_station"]({BERLIN_BBOX});
);
out center;''',

    'restaurant': f'''[out:json][timeout:120];
(
  node["amenity"="restaurant"]({BERLIN_BBOX});
  node["amenity"="fast_food"]({BERLIN_BBOX});
  node["amenity"="pub"]({BERLIN_BBOX});
  node["amenity"="bar"]({BERLIN_BBOX});
);
out center;''',

    'cafe': f'''[out:json][timeout:120];
(
  node["amenity"="cafe"]({BERLIN_BBOX});
);
out center;''',

    'food': f'''[out:json][timeout:120];
(
  node["amenity"="restaurant"]({BERLIN_BBOX});
  node["amenity"="cafe"]({BERLIN_BBOX});
  node["amenity"="fast_food"]({BERLIN_BBOX});
  node["amenity"="pub"]({BERLIN_BBOX});
  node["amenity"="bar"]({BERLIN_BBOX});
);
out center;''',

    'shop': f'''[out:json][timeout:120];
(
  node["shop"="supermarket"]({BERLIN_BBOX});
  node["shop"="convenience"]({BERLIN_BBOX});
  node["shop"="mall"]({BERLIN_BBOX});
  node["shop"="department_store"]({BERLIN_BBOX});
);
out center;''',

    'park': f'''[out:json][timeout:120];
(
  way["leisure"="park"]({BERLIN_BBOX});
  relation["leisure"="park"]({BERLIN_BBOX});
  way["leisure"="garden"]({BERLIN_BBOX});
);
out center;''',

    'water': f'''[out:json][timeout:120];
(
  way["natural"="water"]({BERLIN_BBOX});
  relation["natural"="water"]({BERLIN_BBOX});
  way["waterway"="river"]({BERLIN_BBOX});
  way["waterway"="canal"]({BERLIN_BBOX});
);
out center;''',

    'school': f'''[out:json][timeout:120];
(
  node["amenity"="school"]({BERLIN_BBOX});
  way["amenity"="school"]({BERLIN_BBOX});
  node["amenity"="kindergarten"]({BERLIN_BBOX});
  way["amenity"="kindergarten"]({BERLIN_BBOX});
);
out center;''',

    'building': f'''[out:json][timeout:120];
(
  way["building"]["building"!="no"]({BERLIN_BBOX});
);
out center qt 50000;''',
}

def query_overpass(name, query, max_retries=5):
    """Query Overpass API with caching and retry on 429/504."""
    cache_file = POI_CACHE_DIR / f'{name}.json'
    if cache_file.exists():
        with open(cache_file, encoding='utf-8') as f:
            data = json.load(f)
        print(f'  {name}: {len(data):,} POIs (cached)')
        return data
    
    for attempt in range(max_retries):
        print(f'  {name}: querying Overpass API (attempt {attempt+1})...', end=' ', flush=True)
        try:
            resp = requests.post(OVERPASS_URL, data={'data': query}, timeout=180)
            resp.raise_for_status()
            elements = resp.json().get('elements', [])
            
            pois = []
            for e in elements:
                lat = e.get('lat') or e.get('center', {}).get('lat')
                lon = e.get('lon') or e.get('center', {}).get('lon')
                if lat and lon:
                    pois.append({'lat': lat, 'lon': lon})
            
            with open(cache_file, 'w', encoding='utf-8') as f:
                json.dump(pois, f)
            print(f'{len(pois):,} POIs')
            return pois
        except (requests.exceptions.HTTPError, requests.exceptions.ConnectionError) as e:
            wait = 20 * (attempt + 1)
            print(f'error ({e.__class__.__name__}), waiting {wait}s...')
            _time.sleep(wait)
    
    print(f'  {name}: FAILED after {max_retries} retries — skipping')
    return []

# Download all POI categories with 10s delay between API calls
print('=== Downloading Berlin POIs ===')
poi_data = {}
for i, (name, query) in enumerate(POI_QUERIES.items()):
    poi_data[name] = query_overpass(name, query)
    # Wait 10s between uncached queries to respect Overpass rate limits
    cache_file = POI_CACHE_DIR / f'{name}.json'
    if cache_file.exists() and i < len(POI_QUERIES) - 1:
        # Only wait if we actually just made an API call (file was just created or is not old)
        _time.sleep(10)

print(f'\nTotal POI categories: {len(poi_data)}')
for name, pois in poi_data.items():
    print(f'  {name}: {len(pois):,} POIs')

=== Downloading Berlin POIs ===
  ubahn: 275 POIs (cached)
  transit: 483 POIs (cached)
  restaurant: querying Overpass API (attempt 1)... 9,371 POIs
  cafe: querying Overpass API (attempt 1)... 2,596 POIs
  food: querying Overpass API (attempt 1)... error (HTTPError), waiting 20s...
  food: querying Overpass API (attempt 2)... error (HTTPError), waiting 40s...
  food: querying Overpass API (attempt 3)... error (HTTPError), waiting 60s...
  food: querying Overpass API (attempt 4)... 11,967 POIs
  shop: querying Overpass API (attempt 1)... 2,116 POIs
  park: querying Overpass API (attempt 1)... error (HTTPError), waiting 20s...
  park: querying Overpass API (attempt 2)... error (HTTPError), waiting 40s...
  park: querying Overpass API (attempt 3)... error (HTTPError), waiting 60s...
  park: querying Overpass API (attempt 4)... 6,381 POIs
  water: querying Overpass API (attempt 1)... 2,922 POIs
  school: querying Overpass API (attempt 1)... error (HTTPError), waiting 20s...
  school: que

### A.2 — Build KDTrees & Compute Point-Level OSM Features

Convert lat/lon to approximate meters using the Haversine-based conversion at Berlin's latitude (~52.5°N):
- 1° latitude ≈ 111,320 m
- 1° longitude ≈ 111,320 × cos(52.5°) ≈ 67,700 m

Then build scipy cKDTree for O(log n) nearest-neighbor queries.

In [4]:
# Approximate lat/lon → meters conversion at Berlin's latitude
LAT_TO_M = 111_320  # 1° lat in meters
LON_TO_M = 111_320 * np.cos(np.radians(52.5))  # ~67,700 m at Berlin

def latlon_to_meters(lat, lon):
    """Convert lat/lon arrays to approximate x,y in meters (local flat projection)."""
    return np.column_stack([lon * LON_TO_M, lat * LAT_TO_M])

def build_kdtree(pois):
    """Build a cKDTree from a list of {'lat': ..., 'lon': ...} dicts."""
    coords = np.array([[p['lat'], p['lon']] for p in pois])
    xy = latlon_to_meters(coords[:, 0], coords[:, 1])
    return cKDTree(xy), coords

# Build KDTrees for all POI categories
print('Building KDTrees...')
trees = {}
for name, pois in poi_data.items():
    if len(pois) > 0:
        trees[name] = build_kdtree(pois)
        print(f'  {name}: {len(pois):,} POIs → KDTree built')

# Convert unit coordinates to meters
units_with_coords = units[has_coords].copy()
unit_xy = latlon_to_meters(units_with_coords['lat'].values, units_with_coords['lon'].values)
print(f'\nUnit coordinates converted: {len(unit_xy):,} points')

Building KDTrees...
  ubahn: 275 POIs → KDTree built
  transit: 483 POIs → KDTree built
  restaurant: 9,371 POIs → KDTree built
  cafe: 2,596 POIs → KDTree built
  food: 11,967 POIs → KDTree built
  shop: 2,116 POIs → KDTree built
  park: 6,381 POIs → KDTree built
  water: 2,922 POIs → KDTree built
  school: 3,823 POIs → KDTree built
  building: 50,000 POIs → KDTree built

Unit coordinates converted: 8,250 points


In [5]:
# Compute OSM features for all units
print('Computing OSM features for each unit...')

# --- Distance to CBD (Alexanderplatz: 52.5219, 13.4132) ---
ALEX_LAT, ALEX_LON = 52.5219, 13.4132
alex_xy = latlon_to_meters(np.array([ALEX_LAT]), np.array([ALEX_LON]))
dist_cbd = np.sqrt(((unit_xy[:, 0] - alex_xy[0, 0]) ** 2) + ((unit_xy[:, 1] - alex_xy[0, 1]) ** 2))
units_with_coords['dist_cbd_m'] = dist_cbd
print(f'  dist_cbd_m: median={np.median(dist_cbd):.0f}m, range=[{np.min(dist_cbd):.0f}, {np.max(dist_cbd):.0f}]')

# --- Distance to nearest (multiple POI types) ---
dist_categories = ['transit', 'ubahn', 'park', 'water', 'school']
for name in dist_categories:
    if name in trees:
        tree, _ = trees[name]
        dists, _ = tree.query(unit_xy, k=1)
        units_with_coords[f'dist_{name}_m'] = dists
        print(f'  dist_{name}_m: median={np.median(dists):.0f}m, max={np.max(dists):.0f}m')

# --- Count within radius ---
count_specs = [
    ('food', [500, 1000]),
    ('restaurant', [500, 1000]),
    ('cafe', [500]),
    ('shop', [500, 1000]),
    ('transit', [1000]),
    ('building', [200]),  # building density proxy
]
for name, radii in count_specs:
    if name in trees:
        tree, _ = trees[name]
        for r in radii:
            counts = tree.query_ball_point(unit_xy, r=r, return_length=True)
            col = f'count_{name}_{r}m'
            units_with_coords[col] = counts
            print(f'  {col}: median={np.median(counts):.0f}, max={np.max(counts)}')

print(f'\nOSM features computed for {len(units_with_coords):,} units')

# Quick sanity check: compare with PLZ-level features
osm_plz = pd.read_csv(PROC_DIR / 'spatial_osm_features.csv')
print(f'\n=== Sanity Check: Unit-level vs PLZ-level ===')
for col in ['dist_transit_m', 'count_food_1000m']:
    unit_median = units_with_coords[col].median()
    plz_median = osm_plz[col].median()
    print(f'  {col}: unit median={unit_median:.0f}, PLZ median={plz_median:.0f}')

Computing OSM features for each unit...
  dist_cbd_m: median=6056m, range=[201, 24901]
  dist_transit_m: median=448m, max=4270m
  dist_ubahn_m: median=521m, max=7065m
  dist_park_m: median=174m, max=2211m
  dist_water_m: median=457m, max=1872m
  dist_school_m: median=145m, max=1754m
  count_food_500m: median=28, max=287
  count_food_1000m: median=111, max=639
  count_restaurant_500m: median=21, max=229
  count_restaurant_1000m: median=87, max=477
  count_cafe_500m: median=6, max=65
  count_shop_500m: median=6, max=49
  count_shop_1000m: median=23, max=114
  count_transit_1000m: median=3, max=19
  count_building_200m: median=0, max=162

OSM features computed for 8,250 units

=== Sanity Check: Unit-level vs PLZ-level ===
  dist_transit_m: unit median=448, PLZ median=695
  count_food_1000m: unit median=111, PLZ median=20


## Part B: Satellite Multi-Scale Features

Extract NDVI, NDWI, NDBI from local Sentinel-2 rasters at 3 buffer scales (100m, 250m, 500m) around each unit's coordinates.

In [6]:
# Load Sentinel-2 rasters
RASTERS = {}
for index_name in ['ndvi', 'ndwi', 'ndbi']:
    tif_path = PROC_DIR / f'{index_name}_berlin.tif'
    if tif_path.exists():
        RASTERS[index_name] = rasterio.open(tif_path)
        r = RASTERS[index_name]
        print(f'  {index_name}: {r.width}×{r.height} pixels, CRS={r.crs}, resolution={r.res}')
    else:
        print(f'  {index_name}: FILE NOT FOUND at {tif_path}')

print(f'\nRasters loaded: {len(RASTERS)}')

  ndvi: 10980×10980 pixels, CRS=EPSG:32633, resolution=(10.0, 10.0)
  ndwi: 10980×10980 pixels, CRS=EPSG:32633, resolution=(10.0, 10.0)
  ndbi: 10980×10980 pixels, CRS=EPSG:32633, resolution=(10.0, 10.0)

Rasters loaded: 3


In [7]:
from pyproj import Transformer

def extract_buffer_mean(raster, lat, lon, buffer_m, transformer):
    """Extract mean raster value within a circular buffer around a point.
    
    Args:
        raster: open rasterio dataset
        lat, lon: WGS84 coordinates
        buffer_m: buffer radius in meters
        transformer: pyproj Transformer (WGS84 → raster CRS)
    
    Returns:
        Mean pixel value within buffer, or NaN if out of bounds
    """
    # Transform point to raster CRS
    x, y = transformer.transform(lat, lon)
    
    # Pixel resolution
    res_x, res_y = raster.res
    buffer_pixels = int(buffer_m / res_x) + 1
    
    # Get center pixel
    try:
        row, col = rowcol(raster.transform, x, y)
    except Exception:
        return np.nan
    
    # Check bounds
    if row < 0 or col < 0 or row >= raster.height or col >= raster.width:
        return np.nan
    
    # Read a window around the point
    r_min = max(0, row - buffer_pixels)
    r_max = min(raster.height, row + buffer_pixels + 1)
    c_min = max(0, col - buffer_pixels)
    c_max = min(raster.width, col + buffer_pixels + 1)
    
    window = rasterio.windows.Window.from_slices((r_min, r_max), (c_min, c_max))
    data = raster.read(1, window=window)
    
    # Create circular mask
    rows_idx, cols_idx = np.mgrid[0:data.shape[0], 0:data.shape[1]]
    center_r = row - r_min
    center_c = col - c_min
    dist = np.sqrt(((rows_idx - center_r) * res_x) ** 2 + ((cols_idx - center_c) * abs(res_y)) ** 2)
    mask = (dist <= buffer_m) & (data != 0) & np.isfinite(data)
    
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(data[mask]))

# Set up coordinate transformer (WGS84 → raster CRS)
if RASTERS:
    raster_crs = list(RASTERS.values())[0].crs
    transformer = Transformer.from_crs('EPSG:4326', raster_crs, always_xy=False)
    print(f'Raster CRS: {raster_crs}')
    print(f'Transformer ready: WGS84 → {raster_crs}')
else:
    print('No rasters loaded — skipping satellite features')

Raster CRS: EPSG:32633
Transformer ready: WGS84 → EPSG:32633


In [8]:
# Extract satellite features for all units at 3 buffer scales
BUFFER_RADII = [100, 250, 500]  # meters

if RASTERS:
    n = len(units_with_coords)
    print(f'Extracting satellite features for {n:,} units × {len(RASTERS)} indices × {len(BUFFER_RADII)} scales...')
    print(f'(This may take a few minutes)')
    
    for index_name, raster in RASTERS.items():
        for buffer_m in BUFFER_RADII:
            col_name = f'{index_name}_{buffer_m}m'
            values = []
            for i, (_, row) in enumerate(units_with_coords.iterrows()):
                val = extract_buffer_mean(raster, row['lat'], row['lon'], buffer_m, transformer)
                values.append(val)
                if (i + 1) % 2000 == 0:
                    print(f'  {col_name}: {i+1}/{n}')
            units_with_coords[col_name] = values
            valid = sum(1 for v in values if not np.isnan(v))
            print(f'  {col_name}: done ({valid}/{n} valid, {100*valid/n:.1f}%)')
    
    print(f'\nSatellite features extracted.')
else:
    print('Skipping satellite features (no rasters loaded)')

Extracting satellite features for 8,250 units × 3 indices × 3 scales...
(This may take a few minutes)
  ndvi_100m: 2000/8250
  ndvi_100m: 4000/8250
  ndvi_100m: 6000/8250
  ndvi_100m: 8000/8250
  ndvi_100m: done (8237/8250 valid, 99.8%)
  ndvi_250m: 2000/8250
  ndvi_250m: 4000/8250
  ndvi_250m: 6000/8250
  ndvi_250m: 8000/8250
  ndvi_250m: done (8237/8250 valid, 99.8%)
  ndvi_500m: 2000/8250
  ndvi_500m: 4000/8250
  ndvi_500m: 6000/8250
  ndvi_500m: 8000/8250
  ndvi_500m: done (8237/8250 valid, 99.8%)
  ndwi_100m: 2000/8250
  ndwi_100m: 4000/8250
  ndwi_100m: 6000/8250
  ndwi_100m: 8000/8250
  ndwi_100m: done (8237/8250 valid, 99.8%)
  ndwi_250m: 2000/8250
  ndwi_250m: 4000/8250
  ndwi_250m: 6000/8250
  ndwi_250m: 8000/8250
  ndwi_250m: done (8237/8250 valid, 99.8%)
  ndwi_500m: 2000/8250
  ndwi_500m: 4000/8250
  ndwi_500m: 6000/8250
  ndwi_500m: 8000/8250
  ndwi_500m: done (8237/8250 valid, 99.8%)
  ndbi_100m: 2000/8250
  ndbi_100m: 4000/8250
  ndbi_100m: 6000/8250
  ndbi_100m: 8000/8

## Save `spatial_unit` Table

In [9]:
# Build spatial_unit table — all computed features
osm_cols = [
    # Distance features
    'dist_cbd_m', 'dist_transit_m', 'dist_ubahn_m',
    'dist_park_m', 'dist_water_m', 'dist_school_m',
    # Count features
    'count_food_500m', 'count_food_1000m',
    'count_restaurant_500m', 'count_restaurant_1000m',
    'count_cafe_500m',
    'count_shop_500m', 'count_shop_1000m',
    'count_transit_1000m',
    'count_building_200m',
]

sat_cols = [f'{idx}_{r}m' for idx in ['ndvi', 'ndwi', 'ndbi'] for r in [100, 250, 500]]

# Only include columns that were actually computed
all_feature_cols = [c for c in osm_cols + sat_cols if c in units_with_coords.columns]
spatial_cols = ['unit_id', 'coord_source'] + all_feature_cols

spatial_unit = units_with_coords[spatial_cols].copy()

# Save
spatial_path = PROC_DIR / 'spatial_unit.parquet'
spatial_unit.to_parquet(spatial_path, index=False)
spatial_hash = hashlib.sha256(open(spatial_path, 'rb').read()).hexdigest()

print(f'=== spatial_unit table saved ===')
print(f'File: {spatial_path.name}')
print(f'Rows: {len(spatial_unit):,}')
print(f'Columns ({len(all_feature_cols)} features + 2 meta): {list(spatial_unit.columns)}')
print(f'SHA-256: {spatial_hash[:16]}...')
print()

# Coverage report
print('=== Feature Coverage ===')
for col in all_feature_cols:
    valid = spatial_unit[col].notna().sum()
    print(f'  {col:<28} {valid:>6}/{len(spatial_unit)} ({100*valid/len(spatial_unit):.1f}%)')

print()
n_osm = len([c for c in all_feature_cols if c.startswith('dist_') or c.startswith('count_')])
n_sat = len([c for c in all_feature_cols if any(c.startswith(p) for p in ['ndvi', 'ndwi', 'ndbi'])])

print('=== Phase 3 Complete ===')
print(f'  {len(spatial_unit):,} units with point-level spatial features')
print(f'  {n_osm} OSM features + {n_sat} satellite features = {len(all_feature_cols)} total')
print()
print('=== Data Architecture Status ===')
print(f'  ✅ units:         8,256 rows (with lat/lon + coord_source)')
print(f'  ✅ listings:      8,256 rows (price panel)')
print(f'  ✅ spatial_unit:  {len(spatial_unit):,} rows x {len(all_feature_cols)} features (NEW!)')
print(f'  ✅ spatial_plz:   190 PLZs (legacy fallback)')
print()
print('=== Next: Phase 4 ===')
print('  Retrain XGBoost v4 on 2026 data with unit-level spatial features')

=== spatial_unit table saved ===
File: spatial_unit.parquet
Rows: 8,250
Columns (24 features + 2 meta): ['unit_id', 'coord_source', 'dist_cbd_m', 'dist_transit_m', 'dist_ubahn_m', 'dist_park_m', 'dist_water_m', 'dist_school_m', 'count_food_500m', 'count_food_1000m', 'count_restaurant_500m', 'count_restaurant_1000m', 'count_cafe_500m', 'count_shop_500m', 'count_shop_1000m', 'count_transit_1000m', 'count_building_200m', 'ndvi_100m', 'ndvi_250m', 'ndvi_500m', 'ndwi_100m', 'ndwi_250m', 'ndwi_500m', 'ndbi_100m', 'ndbi_250m', 'ndbi_500m']
SHA-256: e8c253e2a745d99e...

=== Feature Coverage ===
  dist_cbd_m                     8250/8250 (100.0%)
  dist_transit_m                 8250/8250 (100.0%)
  dist_ubahn_m                   8250/8250 (100.0%)
  dist_park_m                    8250/8250 (100.0%)
  dist_water_m                   8250/8250 (100.0%)
  dist_school_m                  8250/8250 (100.0%)
  count_food_500m                8250/8250 (100.0%)
  count_food_1000m               8250/8250